# CV05 — Tín hiệu nhỏ trong video: rPPG

Video ROI da [B,T,3,H,W] và waveform đồng bộ [B,T]. Dùng split theo người. Phân giải tần số phụ thuộc thời lượng cửa sổ, không chỉ FPS.

Năm cell code, chạy lần lượt. Fixture không phải benchmark; xem REFERENCE.md để chọn dữ liệu thật và paper gốc.


## Cell 1 — Môi trường và seed

Dùng CPU để kiểm tra trước. Tìm repo root từ thư mục notebook, đặt seed và import PyTorch. Không tải pretrained weights.


In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn
import torch.nn.functional as F

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'vision/curriculum').is_dir())
sys.path.insert(0, str(ROOT))
torch.manual_seed(42)
torch.set_num_threads(2)
device = torch.device('cpu')  # Change to cuda after the CPU example works.
print('Device:', device)


## Cell 2 — Tạo batch và model

Video ROI da [B,T,3,H,W] và waveform đồng bộ [B,T]. Dùng split theo người. Phân giải tần số phụ thuộc thời lượng cửa sổ, không chỉ FPS. Dữ liệu trong cell là fixture tổng hợp để bạn đọc được shape; thay bằng Dataset thật trước khi so metric.


In [ ]:
from vision.curriculum.cv05_subtle import PulseNet, rgb_trace, chrom, heart_rate, loss_fn
fps = 30
steps = 300
time = torch.arange(steps, device=device) / fps
wave = torch.sin(2 * torch.pi * 1.2 * time)
video = torch.full((2, steps, 3, 8, 8), 0.5, device=device)
video[:, :, 1] += 0.02 * wave[None, :, None, None]
target = wave[None].expand(2, -1)
model = PulseNet().to(device)
batch = (video, target)


## Cell 3 — Forward và baseline

Đọc shape/output trước khi train. Các số in ra chỉ minh họa phép tính và luồng dự đoán.


In [ ]:
green = rgb_trace(video)[..., 1]
print('Green baseline BPM:', heart_rate(green, fps).tolist())
print('Simplified CHROM BPM:', heart_rate(chrom(video), fps).tolist())
print('Target BPM:', 72)


## Cell 4 — Một bước cập nhật

Tính đúng loss của nhiệm vụ, backward và cập nhật tham số. Một bước không phải quy trình huấn luyện hoàn chỉnh. TENT tự thực hiện update bên trong adapt.


In [ ]:
model.train()
optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=3e-4)
optimizer.zero_grad(set_to_none=True)
loss = loss_fn(model, *batch)
loss.backward()
torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
optimizer.step()
print('One-step training loss:', loss.detach().item())


## Cell 5 — Chẩn đoán và hướng nâng cấp

Green trace → CHROM đơn giản → 3D CNN với correlation và frequency loss. Ví dụ có 10 giây video; không dùng 4 frame để suy nhịp tim.


In [ ]:
model.eval()
with torch.no_grad():
    predicted_wave = model(video)
print('Model waveform:', predicted_wave.shape)
print('Untrained/one-step model BPM:', heart_rate(predicted_wave, fps).tolist())
print('Report subject-disjoint HR MAE only after real training.')


## Thử nghiệm tiếp theo

Green trace → CHROM đơn giản → 3D CNN với correlation và frequency loss. Ví dụ có 10 giây video; không dùng 4 frame để suy nhịp tim.

Giữ nguyên split và ngân sách huấn luyện. Đọc [module](../cv05_subtle.py) và [tài liệu gốc](../REFERENCE.md).
